In [0]:
from pyspark.sql.functions import current_timestamp, col

source_path = "/Volumes/bg_traffic/bg_traffic_bronze/landing/openmeteo/"
checkpoint_path = "/Volumes/bg_traffic/bg_traffic_bronze/landing/_checkpoints/openmeteo"
target_table = "bg_traffic.bg_traffic_bronze.openmeteo"

df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/_schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("pathGlobFilter", "*.json")
    .option("multiLine", "true")
    .load(source_path)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_name"))
)

query = (
    df.writeStream.format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)
query.awaitTermination()

In [0]:
display(spark.sql(f"SELECT * FROM {target_table} LIMIT 10"))
